In [ ]:
import os
from pathlib import Path
from typing import List
import json
import gc
import time

import numpy as np
import anndata as ad
from scipy import sparse
from sklearn.decomposition import IncrementalPCA
from tqdm import tqdm
import zarr
from numcodecs import Blosc
import psutil

print("OK: imports")

In [ ]:
import os

INPUT_DIR = Path(os.environ.get("TAHOE_RAW_DIR", "data/tahoe_raw"))
OUTPUT_DIR = Path("tahoe_128")
WORK_DIR = Path("tahoe_work")




N_COMPONENTS = 128
GLOBAL_HVG_TOPK = 6000


SOFT_MEM_LIMIT = 100 * 1024**3   
TARGET_BATCH_MEMORY = 8 * 1024**3  
MIN_BATCH_CELLS = 500             
MAX_BATCH_CELLS = 20000           


CHUNK_ROWS = 20000
COMPRESSOR = Blosc(cname="zstd", clevel=5, shuffle=Blosc.BITSHUFFLE)


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

def list_h5ad_files(folder: Path) -> List[Path]:
    return sorted([p for p in folder.glob("*.h5ad") if p.is_file()])

def rss_bytes() -> int:
    return psutil.Process().memory_info().rss

def human(n: int) -> str:
    for u in ["B","KB","MB","GB","TB","PB"]:
        if n < 1024:
            return f"{n:.2f}{u}"
        n /= 1024
    return f"{n:.2f}EB"

def suggest_batch_cells(n_vars_hvg: int, target_mem_bytes: int = TARGET_BATCH_MEMORY) -> int:
    
    if n_vars_hvg <= 0:
        return 2000
    per_row = n_vars_hvg * 4
    safe_rows = int(target_mem_bytes / max(per_row * 2, 1))
    safe_rows = max(MIN_BATCH_CELLS, min(safe_rows, MAX_BATCH_CELLS))
    return safe_rows

def lower_batch(batch_cells: int) -> int:
    
    return max(MIN_BATCH_CELLS, batch_cells // 2)

def check_soft_limit(phase: str):
    m = rss_bytes()
    if m > SOFT_MEM_LIMIT:
        raise MemoryError(f"[{phase}] RSS {human(m)} exceeds soft memory limit {human(SOFT_MEM_LIMIT)}.")

In [ ]:
files = list_h5ad_files(INPUT_DIR)
if not files:
    raise SystemExit(f"No .h5ad files in input directory: {INPUT_DIR}")

print(f"Found {len(files)} h5ad files.")

var_names = None
gene_var_accum = None

for f in tqdm(files, desc="Pass 1/3: global variance (HVG approx)"):
    A = ad.read_h5ad(f, backed="r")
    if var_names is None:
        var_names = A.var_names.to_numpy()
        gene_var_accum = np.zeros(A.n_vars, dtype=np.float64)

    n_obs = A.n_obs
    
    step = 10000
    s = np.zeros(A.n_vars, dtype=np.float64)
    ss = np.zeros(A.n_vars, dtype=np.float64)
    n = 0

    for i in range(0, n_obs, step):
        Xi = A.X[i:i+step, :]
        if not sparse.issparse(Xi):
            Xi = sparse.csr_matrix(Xi)
        
        Xi = Xi.toarray().astype(np.float32, copy=False)
        s += Xi.sum(axis=0, dtype=np.float64)
        ss += (Xi.astype(np.float64) ** 2).sum(axis=0)
        n += Xi.shape[0]

        
        del Xi
        gc.collect()
        check_soft_limit("HVG-block")

    if n > 0:
        mean = s / n
        var = ss / n - mean**2
        var = np.clip(var, 0, None)
        gene_var_accum += var

    
    try:
        A.file.close()
    except Exception:
        pass
    del A, s, ss
    gc.collect()
    check_soft_limit("HVG-file")


idx = np.argsort(-gene_var_accum)[:GLOBAL_HVG_TOPK]
hvg_names = var_names[idx]
hvg_set = set(hvg_names)
print(f"Approximate global HVG count: {len(hvg_names)}")


np.savez_compressed(WORK_DIR / "hvg_selection.npz",
                    hvg_names=hvg_names,
                    gene_var_accum=gene_var_accum.astype(np.float32))

del gene_var_accum
gc.collect()
check_soft_limit("HVG-done")

In [ ]:
# -*- coding: utf-8 -*-
import os, gc, json, time, sys, traceback
from pathlib import Path
from typing import List, Tuple

import numpy as np
import anndata as ad
from scipy import sparse
from sklearn.decomposition import TruncatedSVD
from tqdm import tqdm
import zarr
from numcodecs import Blosc
import psutil

# =========================

# =========================

INPUT_DIR = Path(os.environ.get("TAHOE_RAW_DIR", "data/tahoe_raw"))
OUTPUT_DIR = Path("tahoe_128")
WORK_DIR = Path("tahoe_work")

HVG_CACHE = WORK_DIR / "hvg_selection.npz"  
MODEL_PATH = WORK_DIR / "tsvd_model.npz"    

N_COMPONENTS = 128


FIT_SAMPLE_TARGET = 500_000  


TRANSFORM_BATCH_ROWS = 50_000


SOFT_MEM_LIMIT = 100 * 1024**3  # 100 GB


CHUNK_ROWS = 20_000
COMPRESSOR = Blosc(cname="zstd", clevel=5, shuffle=Blosc.BITSHUFFLE)


os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "4")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

# =========================

# =========================

def list_h5ad_files(folder: Path) -> List[Path]:
    files = sorted([p for p in folder.glob("*.h5ad") if p.is_file()])
    if not files:
        raise SystemExit(f"Error: No .h5ad files in input directory: {folder}")
    return files

def rss_bytes() -> int:
    return psutil.Process().memory_info().rss

def human(n: int) -> str:
    for u in ["B","KB","MB","GB","TB","PB"]:
        if n < 1024:
            return f"{n:.2f}{u}"
        n /= 1024
    return f"{n:.2f}EB"

def check_soft_limit(phase: str):
    m = rss_bytes()
    if m > SOFT_MEM_LIMIT:
        raise MemoryError(f"[{phase}] RSS {human(m)} exceeds soft memory limit {human(SOFT_MEM_LIMIT)}.")

def to_str_array(x) -> np.ndarray:
    arr = np.asarray(x)
    if arr.dtype.kind in ("S", "a"):  # bytes-like
        arr = arr.astype(str)
    elif arr.dtype == object:
        arr = arr.astype(str)
    return arr

def get_keep_mask(A, hvg_names: np.ndarray) -> np.ndarray:
    
    var_names = to_str_array(A.var_names)
    mask = np.isin(var_names, hvg_names)
    if not mask.any():
        raise ValueError(f"[{getattr(A, 'filename', 'dataset')}] No HVG columns matched; check gene identifiers")
    return mask

def sparse_select_cols_csr(X, keep_mask: np.ndarray):
    
    if not sparse.issparse(X):
        X = sparse.csr_matrix(X)
    else:
        X = X.tocsr()
    X = X[:, keep_mask]
    return X.tocsr()

def open_zarr_for_write(dest_dir: Path, n_rows: int, n_cols: int, chunk_rows: int):
    store = zarr.DirectoryStore(str(dest_dir))
    z = zarr.create(
        shape=(n_rows, n_cols),
        chunks=(chunk_rows, n_cols),
        dtype="float32",
        store=store,
        compressor=COMPRESSOR,
        overwrite=True,
    )
    return z

def safe_close_anndata(A):
    try:
        # anndata>=0.9: A.file is h5py.File-like; older may differ
        if hasattr(A, "file") and A.file is not None:
            A.file.close()
    except Exception:
        pass

def allocate_samples_per_file(files: List[Path], target_total: int) -> Tuple[np.ndarray, np.ndarray]:
    sizes = []
    for f in files:
        A = ad.read_h5ad(f, backed="r")
        sizes.append(A.n_obs)
        safe_close_anndata(A)
        del A
    sizes = np.asarray(sizes, dtype=np.int64)
    total = sizes.sum()
    if total <= 0:
        raise ValueError("Total row count is zero")
    
    alloc = np.maximum(1, np.floor(sizes / total * target_total).astype(int))
    
    diff = int(target_total - alloc.sum())
    order = np.argsort(-sizes)
    k = 0
    while diff != 0 and k < len(files) * 1000:
        idx = int(order[k % len(files)])
        step = 1 if diff > 0 else -1
        newv = alloc[idx] + step
        if 1 <= newv <= sizes[idx]:
            alloc[idx] = newv
            diff -= step
        k += 1
    return alloc, sizes

def load_hvg_names(cache_path: Path) -> np.ndarray:
    if not cache_path.exists():
        raise FileNotFoundError(f"HVG cache not found: {cache_path}")
    data = np.load(cache_path, allow_pickle=True)
    if "hvg_names" not in data:
        raise KeyError(f"{cache_path} does not contain key 'hvg_names'")
    hvg = to_str_array(data["hvg_names"]).ravel()
    if hvg.size == 0:
        raise ValueError("HVG list is empty")
    return hvg

# =========================

# =========================

files = list_h5ad_files(INPUT_DIR)
print(f"Found {len(files)} h5ad files")

hvg_names = load_hvg_names(HVG_CACHE)
print(f"Loaded HVGs: {hvg_names.size} genes from {HVG_CACHE}")

# =========================

# =========================

svd = None
if MODEL_PATH.exists():
    try:
        m = np.load(MODEL_PATH, allow_pickle=True)
        components_ = m["components_"]
        explained_variance_ = m["explained_variance_"]
        explained_variance_ratio_ = m["explained_variance_ratio_"]
        
        svd = TruncatedSVD(n_components=components_.shape[0], algorithm="randomized")
        
        svd.components_ = components_.astype(np.float64, copy=False)
        svd.explained_variance_ = explained_variance_.astype(np.float64, copy=False)
        svd.explained_variance_ratio_ = explained_variance_ratio_.astype(np.float64, copy=False)
        print(f"Loaded fitted model: {MODEL_PATH}; skipping fit")
    except Exception as e:
        print(f"Warning: Could not load existing model; refitting. Reason: {e}")

if svd is None:
    
    print("Allocating SVD fitting sample counts...")
    alloc, sizes = allocate_samples_per_file(files, FIT_SAMPLE_TARGET)
    print(f"- Files: {len(files)}")
    print(f"- Estimated total rows: {int(sizes.sum())}")
    print(f"- Planned sample rows: {int(alloc.sum())}")

    
    X_samples = []
    for f, want in tqdm(list(zip(files, alloc)), desc="Sampling fit matrix"):
        if want <= 0:
            continue
        A = ad.read_h5ad(f, backed="r")
        try:
            keep = get_keep_mask(A, hvg_names)
            n = A.n_obs
            if want >= n:
                rows = np.arange(n, dtype=np.int64)
            else:
                rng = np.random.default_rng(12345)
                rows = np.sort(rng.choice(n, size=int(want), replace=False).astype(np.int64))

            parts = []
            step = 10_000
            for s0 in range(0, rows.size, step):
                idx = rows[s0:s0+step]
                Xi = A.X[idx, :]
                Xi = sparse_select_cols_csr(Xi, keep)
                if Xi.shape[1] != keep.sum():
                    raise RuntimeError(f"{f.name}: Unexpected dimensions after column selection {Xi.shape}")
                parts.append(Xi)
                del Xi
                gc.collect()

            if len(parts) == 0:
                raise RuntimeError(f"{f.name}: No sample blocks collected")
            Xi_all = parts[0] if len(parts) == 1 else sparse.vstack(parts, format="csr")
            if Xi_all.shape[0] == 0 or Xi_all.shape[1] == 0:
                raise RuntimeError(f"{f.name}: Empty sample matrix {Xi_all.shape}")
            X_samples.append(Xi_all)
            del parts, Xi_all
        finally:
            safe_close_anndata(A)
            del A
            gc.collect()
            check_soft_limit("SVD-sample-file")

    if len(X_samples) == 0:
        raise RuntimeError("Cannot fit SVD: sample collection is empty")

    X_sample = X_samples[0] if len(X_samples) == 1 else sparse.vstack(X_samples, format="csr")
    del X_samples
    gc.collect()
    check_soft_limit("SVD-sample-assembled")

    print(f"Sample matrix shape: {X_sample.shape}, nnz={X_sample.nnz}")
    if X_sample.shape[0] < N_COMPONENTS:
        raise ValueError(f"Sample row count {X_sample.shape[0]} is less than n_components={N_COMPONENTS}; increase FIT_SAMPLE_TARGET")

    
    svd = TruncatedSVD(
        n_components=N_COMPONENTS,
        algorithm="randomized",
        n_iter=5,
        random_state=12345
    )
    t0 = time.time()
    svd.fit(X_sample)
    t1 = time.time()
    print(f"TruncatedSVD fit finished in {t1 - t0:.1f}s, total explained variance ratio={svd.explained_variance_ratio_.sum():.4f}")

    
    np.savez_compressed(
        MODEL_PATH,
        components_=svd.components_.astype(np.float32),
        explained_variance_=svd.explained_variance_.astype(np.float32),
        explained_variance_ratio_=svd.explained_variance_ratio_.astype(np.float32),
        hvg_names=hvg_names,
        algo=np.array(["TruncatedSVD(randomized)"], dtype=object),
        n_iter=np.array([5], dtype=np.int32),
        random_state=np.array([12345], dtype=np.int32),
    )
    print(f"Model saved: {MODEL_PATH}")

    
    del X_sample
    gc.collect()

# =========================

# =========================

errors = []
for f in tqdm(files, desc="Pass 2/2: transform + write zarr"):
    out_dir = OUTPUT_DIR / f"{f.stem}.X{N_COMPONENTS}.zarr"
    meta_path = OUTPUT_DIR / f"{f.stem}.X{N_COMPONENTS}.meta.json"

    if out_dir.exists() and meta_path.exists():
        print(f"Skipping: Already exists: {out_dir.name}")
        continue

    A = ad.read_h5ad(f, backed="r")
    try:
        keep = get_keep_mask(A, hvg_names)
        n_obs = A.n_obs
        if n_obs == 0:
            print(f"Warning: Empty file: {f.name}; skipping")
            continue

        Z = open_zarr_for_write(out_dir, n_rows=n_obs, n_cols=N_COMPONENTS, chunk_rows=CHUNK_ROWS)

        row = 0
        for i in range(0, n_obs, TRANSFORM_BATCH_ROWS):
            j = min(i + TRANSFORM_BATCH_ROWS, n_obs)
            Xi = A.X[i:j, :]
            Xi = sparse_select_cols_csr(Xi, keep)
            Zi = svd.transform(Xi).astype(np.float32, copy=False)
            if Zi.shape[1] != N_COMPONENTS:
                raise RuntimeError(f"{f.name}: Unexpected transformed dimensions {Zi.shape}")
            Z[row:row+Zi.shape[0], :] = Zi
            row += Zi.shape[0]
            del Xi, Zi
            gc.collect()
            check_soft_limit("transform-batch")

        meta = {
            "source_file": str(f),
            "n_rows": int(n_obs),
            "n_components": int(N_COMPONENTS),
            "chunk_rows": int(CHUNK_ROWS),
            "compressor": "blosc+zstd",
            "global_hvg": int(hvg_names.size),
            "model_path": str(MODEL_PATH),
            "hvg_list_saved": True,
            "method": "TruncatedSVD(randomized)"
        }
        with open(meta_path, "w", encoding="utf-8") as mf:
            json.dump(meta, mf, ensure_ascii=False, indent=2)

        print(f"Completed: {f.name} -> {out_dir.name}")
    except Exception as e:
        
        err = f"{f.name}: {e}"
        print(f"Error: {err}")
        traceback.print_exc(file=sys.stdout)
        errors.append(err)
        
        try:
            if out_dir.exists():
                for child in Path(out_dir).glob("**/*"):
                    pass  
                
                import shutil
                shutil.rmtree(out_dir, ignore_errors=True)
        except Exception:
            pass
    finally:
        safe_close_anndata(A)
        del A
        gc.collect()
        check_soft_limit("transform-file")

# =========================

# =========================

if errors:
    print("\nSummary: The following files failed (others completed or were skipped):")
    for e in errors:
        print(" -", e)
    raise SystemExit("Some files failed; review errors and retry. Existing completed files will be skipped.")
else:
    print("\nAll files processed")

In [ ]:
import os
# -*- coding: utf-8 -*-
import gc
from pathlib import Path
import numpy as np
import pandas as pd
import anndata as ad
import zarr
from numcodecs import Blosc
from scipy import sparse


INPUT_DIR = Path(os.environ.get("TAHOE_RAW_DIR", "data/tahoe_raw"))
EMB_DIR = Path("tahoe_128")  
N_COMPONENTS = 128
OUT_PATH = Path(f"tahoe_merged_X{N_COMPONENTS}.h5ad")


TMP_DIR = Path("tahoe_merge_work")
TMP_DIR.mkdir(parents=True, exist_ok=True)
OBSM_TMP = TMP_DIR / f"obsm_X_svd{N_COMPONENTS}.zarr"
EMB_CHUNK = 20000  

COMPRESSOR = Blosc(cname="zstd", clevel=5, shuffle=Blosc.BITSHUFFLE)

def list_pairs(input_dir: Path, emb_dir: Path, n_components: int):
    pairs = []
    for f in sorted(input_dir.glob("*.h5ad")):
        z = emb_dir / f"{f.stem}.X{n_components}.zarr"
        if z.exists():
            pairs.append((f, z))
        else:
            print(f"Warning: Missing embeddings: {z}")
    if not pairs:
        raise SystemExit("Error: No embedding Zarr arrays found")
    return pairs

pairs = list_pairs(INPUT_DIR, EMB_DIR, N_COMPONENTS)


row_counts = []
n_total = 0
for f, zpath in pairs:
    z = zarr.open(zpath, mode="r")
    n = int(z.shape[0])
    if z.shape[1] != N_COMPONENTS:
        raise RuntimeError(f"{zpath.name}: Dimension mismatch {z.shape}")
    row_counts.append((f, zpath, n))
    n_total += n

print(f"Total cells to merge: {n_total}")


if OBSM_TMP.exists():
    
    import shutil
    shutil.rmtree(OBSM_TMP, ignore_errors=True)
store = zarr.DirectoryStore(str(OBSM_TMP))
emb_all = zarr.create(
    shape=(n_total, N_COMPONENTS),
    chunks=(EMB_CHUNK, N_COMPONENTS),
    dtype="float32",
    store=store,
    compressor=COMPRESSOR,
    overwrite=True,
)


obs_list = []
row_pos = 0
for f, zpath, n in row_counts:
    
    A = ad.read_h5ad(f, backed="r")
    try:
        obs = A.obs.copy()  
        if len(obs) != n:
            raise RuntimeError(f"{f.name}: obs row count {len(obs)} differs from embedding row count {n}")
        obs["batch"] = f.stem
        obs_list.append(obs)
    finally:
        try:
            A.file.close()
        except Exception:
            pass
        del A
        gc.collect()

    
    z = zarr.open(zpath, mode="r")
    written = 0
    while written < n:
        k = min(EMB_CHUNK, n - written)
        block = z[written:written+k, :]
        
        if block.dtype != np.float32:
            block = block.astype(np.float32, copy=False)
        emb_all[row_pos:row_pos+k, :] = block
        row_pos += k
        written += k
        del block
        gc.collect()


obs = pd.concat(obs_list, axis=0)
obs.index.name = "cell"
del obs_list
gc.collect()




X_placeholder = sparse.csr_matrix((n_total, 0), dtype=np.float32)
adata = ad.AnnData(X_placeholder, obs=obs, var=pd.DataFrame(index=[]))


emb_np = zarr.open(emb_all.store, mode="r")[:]  
adata.obsm[f"X_svd{N_COMPONENTS}"] = emb_np  # float32
del emb_np
gc.collect()






adata.write_h5ad(OUT_PATH, compression="gzip")
print(f"Completed: Saved: {OUT_PATH}")
print(f" - Cells: {adata.n_obs}")
print(f" - obsm['X_svd{N_COMPONENTS}']: {adata.obsm[f'X_svd{N_COMPONENTS}'].shape}, dtype={adata.obsm[f'X_svd{N_COMPONENTS}'].dtype}")
print(f" - X shape: {adata.shape}(zero-column placeholder)")
print(f" - Temporary embedding Zarr: {OBSM_TMP} may be removed when no longer needed")